# real-chart-bench: LineFormer pretrained baseline (Google Colab)

Runs the LineFormer pretrained model (ICDAR2023, arxiv 2305.01837) against the
**real-chart-bench** v0 verified-image evaluation suite and writes
`results/lineformer-pretrained.json` in the project's standard schema.

**Why Colab, not local**: `mmcv` ships source-only on PyPI and OpenMMLab's
prebuilt wheels are Linux+CUDA only. That is exactly what a Colab GPU runtime
provides for free, so this notebook targets Colab instead of fighting a local
macOS/Python-3.14 environment (see `docs/design/benchmark-architecture.md`
§7.16 for the local-infeasibility writeup, §7.19 for this notebook's context).

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (or better). CPU also works but is slow.
2. The repo (`t29mato/real-chart-bench`) is **public** — no GitHub token needed.
3. Run cells **top to bottom, in order, in a single session** — no kernel
   restart is needed anywhere in this notebook (design §7.35: LineFormer runs
   in its own isolated Python 3.10 environment via `uv`, invoked as a
   subprocess, precisely so this notebook's own kernel never needs to change
   Python version). Cell 2 installs `real_chart_bench` with a regular
   (non-editable) `pip install .` — editable installs (`-e .`) do NOT
   reliably import in the same kernel session without a restart (confirmed
   failure mode, fixed 2026-08-19, see design §7.26) — and prints an explicit
   `OK: real_chart_bench installed at ...` confirmation. If that line doesn't
   print, don't run anything below it; re-run Cell 2 (Runtime → Restart
   session first, if it still fails).
4. Estimated free-tier runtime: 5–20 minutes (dominated by building the
   isolated Python 3.10 environment + mmcv/mmdet install, not inference — the
   eval set is tiny, see the verification-gate note below).

**Scope note**: real-image evaluation is gated on `data/verified_pairs/registry.json`
(§7.19: "量より信頼性。ベンチマークの信用が資産" — reliability over quantity). Only
`status: "verified"` entries are used; this is intentionally a small, trustworthy
set rather than the full (unverified) image↔ground-truth pairing.

## 0. Why LineFormer runs in an isolated Python 3.10 environment

**Root cause of the Colab run #7 failure**: `pip install torch==1.13.1+cu117`
could not resolve at all (candidates offered were `2.5.0` and newer only).
Colab's default runtime is now **Python 3.12**, and PyTorch never published a
`cp312` wheel for the `1.13.1` series — verified directly against
`https://download.pytorch.org/whl/cu117/torch/`, which lists `cp37`–`cp311`
wheels for `torch-1.13.1+cu117` and nothing newer. No index URL or flag can
make a wheel exist that was never built. `mmcv_full==1.7.2` has the same
constraint (checked against OpenMMLab's own wheel index,
`https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html` —
`cp310`/`cp311` wheels exist, no `cp312`). LineFormer's own
[`install.sh`](https://github.com/TheJaeLal/LineFormer/blob/main/install.sh)
already assumes this constraint — it starts with `conda create --name
LineFormer python=3.8` for exactly this reason.

**Approach considered and rejected: `condacolab`.** The obvious way to pin a
different Python version inside Colab looked like
[`condacolab`](https://github.com/conda-incubator/condacolab)
(`condacolab.install(python_version="3.10")`). Verified locally (`pip install
condacolab`, then read the installed source directly — not the GitHub `main`
branch docs, which describe an unreleased rewrite that doesn't match what
ships): the **published** package (`0.1.12`, current as of 2026-08-26)
hardcodes `TARGET_PYTHON = "3.12"` and its `_check_python()` step *asserts*
Colab's current interpreter already equals that value before doing anything.
It has no `python_version` parameter, and it doesn't swap the interpreter at
all — it only bolts a matching-ABI conda `site-packages` directory onto
Colab's *existing* Python via `sys.path`/`PATH` patching. In other words,
`condacolab` cannot deliver a Python 3.10 kernel; it requires one that's
already whatever version Colab currently ships (3.12), which is exactly the
version that has no `torch==1.13.1` wheel. It's the wrong tool for this
problem, not a viable fallback.

**Approach considered and rejected: migrate to mmdet 3.x / mmcv 2.x (torch
2.x).** LineFormer's repo vendors a modified copy of `mmdetection` written
against the **mmdet 2.x** config/registry system, and its pretrained
checkpoint (`iter_3000.pth`) was trained against that architecture's exact
module layout. mmdet 3.x rewrote both the config format and the model
registry as a breaking change — porting would mean rewriting LineFormer's
vendored configs from scratch *and* somehow reconciling the old checkpoint's
parameter names with the new architecture, with no official conversion path
provided by either LineFormer or OpenMMLab. This is a research-grade
undertaking with no confidence of even converging, not a notebook fix.

**Chosen approach**: keep the notebook's own Colab kernel on whatever Python
Colab ships (no restart, no fragile interpreter-swapping trick needed at
all) and instead give **LineFormer its own fully isolated Python 3.10
environment on disk**, built with [`uv`](https://docs.astral.sh/uv/)
(`uv python install 3.10` downloads a standalone, self-contained CPython
build — verified locally end-to-end: install, `uv venv --python 3.10`, and
`uv pip install` into it all work exactly as documented, with zero
dependency on the host's system Python or any apt/conda repository). Cell 2
below builds that environment and installs `torch==1.13.1+cu117` +
`mmcv-full` + LineFormer's vendored `mmdetection` into it. The
`LineFormerModelRunner` (Cell 5) then runs LineFormer inference as a
**subprocess** in that Python 3.10 environment — the notebook's own kernel
process never imports `torch`/`mmcv`/`mmdet` directly, so there's nothing
for a Python-version mismatch to break, and no automatic kernel restart is
needed anywhere in this notebook. Trade-off, stated honestly: each
`extract()` call reloads the model weights in a fresh subprocess (~5–10s
overhead per figure) instead of loading once and reusing — a deliberate
simplicity/reliability choice (design §7.35) over a persistent-worker
protocol, and it also means one figure's inference failure can never corrupt
or block any other figure's evaluation.

## 1. Clone real-chart-bench and install it + eval-only deps

In [ ]:
# real-chart-bench is now PUBLIC — no token needed.
!rm -rf /content/real-chart-bench
!git clone --depth 1 https://github.com/t29mato/real-chart-bench.git /content/real-chart-bench
%cd /content/real-chart-bench

# Regular install, NOT editable (-e). Editable installs (PEP 660) work by
# writing a .pth file that maps back to src/ -- but Python's `site` module
# only processes .pth files at *interpreter startup*. A Colab/Jupyter kernel
# is one long-lived interpreter, so `%pip install -e .` succeeds and then
# `import real_chart_bench` in the SAME session still raises
# ModuleNotFoundError (confirmed twice by the owner; reproduced locally in a
# clean venv while fixing this cell -- see design §7.19/§7.26). A regular
# install copies the package straight into site-packages, which is on
# sys.path immediately, no restart needed. This repo doesn't need live-edit
# semantics here (it's a fresh git clone per run), so there's no downside.
#
# %pip (not !pip) installs into the *running kernel's* environment. In some
# Colab configurations !pip can target a different interpreter than the
# kernel, which is a separate, unrelated failure mode this also guards
# against.
%pip install -q .
%pip install -q pymupdf requests

# Fail loudly, right here, if the install didn't actually take -- instead of
# a confusing ModuleNotFoundError several cells downstream. If this cell
# doesn't print the OK line, do not run anything below it.
import importlib

importlib.invalidate_caches()
import real_chart_bench  # noqa: F401

print(f"OK: real_chart_bench installed at {real_chart_bench.__file__}")


## 2. Build LineFormer's isolated Python 3.10 environment + install its stack

Everything in this cell targets a **separate Python 3.10 interpreter on
disk** (`/content/lineformer_venv`), built with `uv` — never this notebook's
own kernel process (see Cell 0 for why). `LineFormerModelRunner` (Cell 5)
invokes that interpreter as a subprocess for each figure.

**Root cause of the Colab run #5 failure** (`ModuleNotFoundError: No module
named 'mmcv'`): an earlier version of this cell installed the **wrong**
OpenMMLab generation — `mmcv>=2.0.0` + `mmengine` + PyPI `mmdet` (the *new*
mmcv-2.x/mmdet-3.x line). `TheJaeLal/LineFormer`'s own
[`install.sh`](https://github.com/TheJaeLal/LineFormer/blob/main/install.sh)
requires the **old** line instead: PyPI package **`mmcv-full`** (mmcv 1.x)
via `mim install mmcv-full`, plus LineFormer's own **vendored**
`mmdetection/` directory (confirmed present at the repo root, confirmed NOT a
git submodule) installed with `pip install -e mmdetection` — not the PyPI
`mmdet` package.

**Root cause of the Colab run #7 failure** (`pip` couldn't resolve
`torch==1.13.1+cu117` at all): Colab's default Python moved to 3.12, and that
torch series was never built for `cp312` — see Cell 0 for the full
investigation and why this cell now targets an isolated Python 3.10
environment instead of the notebook's own kernel.

**Verified against source, not yet executed on Colab hardware**: the install
commands below and the `LineFormerModelRunner`/worker-script API calls in
Cell 5 were checked against LineFormer's actual `install.sh` and `README.md`
(fetched 2026-08-25/26) — including the `infer.load_model(config, checkpoint,
device)` 3-argument signature. This cell verifies its own install immediately
(import checks against the Python 3.10 environment specifically, with
explicit pass/fail diagnostics) so that if anything is still misaligned, the
*next* run fails loudly right here instead of several cells downstream.

In [ ]:
# --- 2a. uv builds a standalone, self-contained Python 3.10 (no apt/conda
# repo dependency -- verified locally end-to-end on 2026-08-26: `uv python
# install 3.10` + `uv venv --python 3.10` + `uv pip install` all work exactly
# as documented). This never touches the notebook kernel's own interpreter.
%pip install -q uv
!uv python install 3.10

PY310_VENV = "/content/lineformer_venv"
!uv venv --python 3.10 -q {PY310_VENV}
PY310 = f"{PY310_VENV}/bin/python"

# `uv venv` deliberately doesn't seed pip/setuptools by default (uv manages
# installs itself). Several packages in this legacy OpenMMLab-era stack
# still import the long-deprecated `pkg_resources` at *runtime* (confirmed
# locally: this is exactly why `openmim`/`mim` crashes in a bare uv venv --
# see 2c below, which avoids `mim` entirely for that reason). Current
# setuptools (verified locally: 84.0.0) no longer ships `pkg_resources` at
# all; `setuptools<81` is confirmed locally to still include it.
!uv pip install -q --python {PY310} "setuptools<81"

# --- 2b. Pin torch/CUDA to the combo LineFormer's install.sh was authored
# against (PyTorch 1.13.1 + CUDA 11.7), into the isolated env above --
# cp310 wheels exist for this exact combo (verified against
# download.pytorch.org/whl/cu117/torch/, and resolved end-to-end with `uv
# pip install --dry-run --python-platform linux`, 2026-08-26).
!uv pip install -q --python {PY310} torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117

# --- 2c. mmcv-full (mmcv 1.x), NOT `mmcv` (mmcv 2.x) -- see Cell 4 markdown
# for the run #5 root cause this fixes. Installed by pointing pip directly
# at OpenMMLab's own prebuilt-wheel index for this exact torch+CUDA combo
# (`-f ...index.html`), NOT via `mim install mmcv-full`: verified locally
# that `mim` crashes with `ModuleNotFoundError: No module named
# 'pkg_resources'` in a bare `uv`-built venv without the shim above. Going
# straight to the wheel index sidesteps `mim` entirely -- also verified via
# `uv pip install --dry-run --python-platform linux`, which fully resolved
# mmcv-full==1.7.2 plus its transitive deps (addict, yapf, ...) from this
# exact URL, 2026-08-26.
!uv pip install -q --python {PY310} mmcv-full==1.7.2 \
    -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html

!git clone --depth 1 https://github.com/TheJaeLal/LineFormer.git /content/LineFormer

# --- 2d. mmdetection/ is a real, plain directory committed inside the
# LineFormer repo (confirmed via the GitHub API file listing and a 404 on
# .gitmodules -- NOT a git submodule, so the --depth 1 clone above already
# has it). It's LineFormer's own modified copy, pinned to work with
# mmcv-full -- installing PyPI `mmdet` instead pulls in the incompatible
# mmdet 3.x line. Editable install is fine here (no PEP 660 staleness risk
# like Cell 2's real_chart_bench install -- this env is only ever used via
# fresh `subprocess` calls, never imported into a long-lived kernel).
# NOTE: this specific step (building LineFormer's vendored mmdetection
# itself, which may compile C extensions) is the one part of this cell that
# could not be dry-run-verified from this environment -- macOS can't build
# it, and there's no cross-platform "would this compile" check. If this
# step fails, the error will be in this cell's own output, not several
# cells downstream.
!uv pip install -q --python {PY310} -e /content/LineFormer/mmdetection

# --- 2e. Remaining plain-pip deps, transcribed 1:1 from install.sh (there is
# no requirements.txt in this repo).
!uv pip install -q --python {PY310} chardet scikit-image matplotlib \
    opencv-python pillow scipy==1.9.3 bresenham tqdm

# --- 2f. Verify immediately, in this same cell, instead of failing several
# cells downstream with an opaque ModuleNotFoundError (the exact failure
# mode from Colab run #5). Checks run *in the PY310 subprocess*, not this
# kernel -- `import mmcv` here in the kernel would always fail regardless of
# whether the install succeeded, since mmcv-full only exists in PY310_VENV.
import subprocess


def _check(label: str, code: str) -> bool:
    result = subprocess.run(
        [PY310, "-c", code],
        cwd="/content/LineFormer",
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print(f"OK: {label} (version={result.stdout.strip()})")
        return True
    tail = (result.stderr.strip().splitlines() or ["(no stderr)"])[-1]
    print(f"FAILED: {label} -- {tail}")
    return False


_checks_ok = True
_checks_ok &= _check(
    "torch", "import torch; print(torch.__version__)"
)
_checks_ok &= _check(
    "mmcv (mmcv-full)", "import mmcv; print(mmcv.__version__)"
)
_checks_ok &= _check(
    "mmdet (LineFormer's vendored mmdetection/)",
    "import mmdet; print(mmdet.__version__)",
)
_checks_ok &= _check(
    "infer (LineFormer's own module)",
    "import sys; sys.path.insert(0, '/content/LineFormer'); import infer; print('ok')",
)

if not _checks_ok:
    raise RuntimeError(
        "LineFormer stack install verification FAILED -- see the FAILED "
        "line(s) above for which module and why. Do not run anything below "
        "this cell until every check prints OK. Common cause: Colab's GPU "
        "driver/CUDA build has drifted from the pinned torch==1.13.1+cu117 "
        "above (mmcv-full's wheel is torch-version-specific) -- check the "
        "torch version printed above against "
        "https://download.pytorch.org/whl/cu117 for an available wheel."
    )
print("\nAll LineFormer/mmcv install checks passed (Python 3.10 subprocess env).")

# --- 2g. Pretrained checkpoint. LineFormer's README does not offer a direct
# file URL -- only a Google Drive *folder* link -- so this uses gdown's
# --folder mode. Downloaded via the notebook kernel's own Python (gdown has
# no torch/mmcv dependency, so it doesn't need the PY310 env).
%pip install -q gdown

CHECKPOINT_DIR = "/content/LineFormer/checkpoints"
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/iter_3000.pth"
CONFIG_PATH = "/content/LineFormer/lineformer_swin_t_config.py"
CHECKPOINT_FOLDER_URL = (
    "https://drive.google.com/drive/folders/1K_zLZwgoUIAJtfjwfCU5Nv33k17R0O5T"
)

import pathlib

pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
if not pathlib.Path(CHECKPOINT_PATH).exists():
    !gdown --folder -q --continue -O {CHECKPOINT_DIR} "{CHECKPOINT_FOLDER_URL}"
    # gdown --folder preserves the Drive folder's own subfolder layout under
    # CHECKPOINT_DIR, so the checkpoint may land one level deeper than
    # CHECKPOINT_PATH -- find it by filename and move it into place rather
    # than assuming the exact path.
    if not pathlib.Path(CHECKPOINT_PATH).exists():
        found = list(pathlib.Path(CHECKPOINT_DIR).rglob("iter_3000.pth"))
        if found:
            found[0].rename(CHECKPOINT_PATH)

if pathlib.Path(CHECKPOINT_PATH).exists():
    size_mb = pathlib.Path(CHECKPOINT_PATH).stat().st_size / 1e6
    print(f"OK: checkpoint present at {CHECKPOINT_PATH} ({size_mb:.1f} MB)")
else:
    raise RuntimeError(
        f"Checkpoint download failed -- {CHECKPOINT_PATH} does not exist "
        f"after gdown. The Drive folder link may have moved; check "
        f"https://github.com/TheJaeLal/LineFormer#pretrained-models for the "
        f"current link and update CHECKPOINT_FOLDER_URL above."
    )

## 3. Get the verified-pairs images

**Bundled-first (design §7.33)**: every current VERIFIED registry entry's image is
committed directly under `data/verified_pairs/images/` or `data/verified_pairs/crops/`
(CC BY 4.0, attribution in `data/verified_pairs/ATTRIBUTION.md`) — so after the
`git clone` in Cell 2, these are already on disk, no network fetch needed. A bare-filename
`image_path` (not currently used by any entry, but a valid registry shape for a future
non-bundled addition) falls back to a live PDF re-fetch via OpenAlex→publisher.

**Why this matters**: an earlier version of this notebook *always* re-fetched every
image live on every run. That failed on the owner's 4th Colab run
(`PdfFetchStatus.NOT_A_PDF` for paper 4173) — the same URL fetches a valid PDF from a
residential/office IP (confirmed locally) but not from Colab's network, most likely
because publishers commonly block known cloud/datacenter IP ranges as anti-scraping
measures. Re-fetching on every run made the whole notebook fragile to that kind of
transient, Colab-specific network blocking for *any* paper, not just 4173. Bundling
removes that dependency entirely for every currently-committed pairing.

**Skip-and-report, not crash-and-die**: if a bare-filename entry's live re-fetch still
fails for some future entry, this cell logs it as **skipped** (with a reason) and moves
on — it does not raise and kill the whole run. The final results explicitly state how
many pairings were evaluated vs. skipped, and the score is computed only over the
evaluated ones.

In [ ]:
%cd /content/real-chart-bench
import json
import pathlib

from real_chart_bench.adapter.verified_pairing_registry import load_registry
from real_chart_bench.usecase.real_image_gate import select_verified_pairings

REPO_ROOT = pathlib.Path("/content/real-chart-bench")
registry = load_registry(REPO_ROOT / "data/verified_pairs/registry.json")
verified = select_verified_pairings(registry)
print(f"{len(verified)} VERIFIED pairing(s) will be evaluated:")
for p in verified:
    print(f"  paper {p.paper_id}, figure {p.figure_id}, panel {p.panel_label!r}")

In [ ]:
# papers.json does not store pdf_url (it's small committed metadata, design
# data-layout convention) -- only needed for the live-refetch fallback path.
import urllib.parse
import urllib.request

from real_chart_bench.adapter.pdf_fetch import HttpPdfFetchAdapter
from real_chart_bench.adapter.figure_extraction import PyMuPdfFigureExtractor
from real_chart_bench.usecase.pdf_fetch import PdfFetchStatus

papers = json.loads((REPO_ROOT / "data/manifest/v0/papers.json").read_text())
papers_by_id = {p["paper_id"]: p for p in papers}


def resolve_pdf_url(doi: str) -> str | None:
    params = urllib.parse.urlencode({"filter": f"doi:{doi}", "per-page": 1})
    req = urllib.request.Request(
        f"https://api.openalex.org/works?{params}",
        headers={"User-Agent": "real-chart-bench/0.0.1 (mailto:tomoya.matou@gmail.com)"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.load(resp)
    results = data.get("results") or []
    if not results:
        return None
    work = results[0]
    best_oa = (work.get("best_oa_location") or {})
    primary = (work.get("primary_location") or {})
    return best_oa.get("pdf_url") or primary.get("pdf_url")


def resolve_image_path(pairing) -> pathlib.Path:
    # Mirrors scripts/eval/run_baselines.py's _resolve_image_path (design
    # §7.21/§7.33): a '/' in image_path means a committed, repo-relative
    # asset (bundled image or manual crop) -- already present after the
    # git clone in Cell 2, no fetch needed. A bare filename is the
    # data/raw/images/{paper_id}/ convention, not currently used by any
    # entry, handled by the live-refetch fallback below.
    if "/" in pairing.image_path:
        return REPO_ROOT / pairing.image_path
    return REPO_ROOT / "data/raw/images" / pairing.paper_id / pairing.image_path


pdf_fetcher = HttpPdfFetchAdapter()
extractor = PyMuPdfFigureExtractor()

images_by_pairing: dict[tuple[str, str], bytes] = {}
skipped: list[dict] = []

for pairing in verified:
    key = (pairing.paper_id, pairing.figure_id)
    bundled_path = resolve_image_path(pairing)
    if bundled_path.exists():
        images_by_pairing[key] = bundled_path.read_bytes()
        continue

    # Fallback: live re-fetch, only reached for a bare-filename entry with
    # no committed image. Skip-and-report (design §7.33), not
    # crash-and-die: a single publisher blocking Colab's network (as
    # happened for paper 4173, see Cell 5) must not kill the whole run.
    try:
        paper = papers_by_id[pairing.paper_id]
        pdf_url = resolve_pdf_url(paper["doi"])
        if not pdf_url:
            raise RuntimeError(f"no pdf_url resolvable (doi={paper['doi']})")

        fetch_result = pdf_fetcher.fetch(pdf_url)
        if fetch_result.status is not PdfFetchStatus.OK or not fetch_result.content:
            raise RuntimeError(f"PDF fetch failed: {fetch_result.status}")

        extracted = extractor.extract(fetch_result.content)
        named = {}
        for i, img in enumerate(extracted):
            ext = "png" if img.source.value == "page_render" else "jpg"
            named[f"p{img.page_number:02d}_{img.source.value}_{i}.{ext}"] = img.image_bytes
        if pairing.image_path not in named:
            raise RuntimeError(
                f"expected image {pairing.image_path!r} not found among "
                f"{len(named)} re-extracted images"
            )
        images_by_pairing[key] = named[pairing.image_path]
    except Exception as exc:  # noqa: BLE001 -- one bad fetch must not kill the whole run
        skipped.append({"paper_id": pairing.paper_id, "figure_id": pairing.figure_id, "reason": str(exc)})
        print(f"SKIPPED paper {pairing.paper_id} figure {pairing.figure_id}: {exc}")

print(f"\n{len(images_by_pairing)} image(s) available, {len(skipped)} pairing(s) skipped")


## 4. Build the DatasetItems (real verified pairs + synthetic fixtures)

Mirrors `scripts/eval/run_baselines.py` exactly, so LineFormer's results are
directly comparable to the naive-CV baseline on the same figures.

In [ ]:
from real_chart_bench.adapter.panel_layout import PyMuPdfPanelSplitter
from real_chart_bench.domain.curve import Curve, ScaleType
from real_chart_bench.usecase.evaluate_dataset import DatasetItem
from real_chart_bench.usecase.model_runner import ExtractionTask

# data/manifest/v0/curves.json only has metadata (n_points, series_label,
# ...), not the actual x/y values -- those live in
# data/cache/ThermoelectricMaterials_curves.csv.gz, which is gitignored
# (large, regeneratable) and therefore absent after a fresh git clone.
# data/verified_pairs/ground_truth.json is the committed subset actually
# needed (just the ~30 figure_ids the registry references, design §7.33 --
# same "bundle only what evaluation needs" pattern as the images above).
ground_truth_by_figure = json.loads((REPO_ROOT / "data/verified_pairs/ground_truth.json").read_text())


def ground_truth_for(figure_id: str) -> list[Curve]:
    return [
        Curve(x_values=tuple(r["x"]), y_values=tuple(r["y"]), series_label=r["prop_y"])
        for r in ground_truth_by_figure.get(figure_id, [])
        if r["x"]  # some Starrydata rows are empty digitization artifacts (n_points=0)
    ]


real_items = []
splitter = PyMuPdfPanelSplitter()
for pairing in verified:
    key = (pairing.paper_id, pairing.figure_id)
    if key not in images_by_pairing:
        continue  # skipped in Cell 7 -- excluded from scoring, not a crash
    image_bytes = images_by_pairing[key]
    if pairing.panel_label is not None:
        panels = {p.label: p for p in splitter.split(image_bytes)}
        image_bytes = panels[pairing.panel_label].image_bytes
    task = ExtractionTask(
        image_bytes=image_bytes,
        x_range=pairing.x_range,
        y_range=pairing.y_range,
        x_scale=pairing.x_scale,
        y_scale=pairing.y_scale,
    )
    real_items.append(
        DatasetItem(
            figure_id=f"{pairing.paper_id}-{pairing.figure_id}",
            task=task,
            ground_truth=ground_truth_for(pairing.figure_id),
        )
    )

print(f"{len(real_items)} real DatasetItem(s) built ({len(skipped)} pairing(s) skipped, see Cell 7)")

In [ ]:
import pymupdf


def synthetic_items() -> list[DatasetItem]:
    items = []

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(1, 0, 0), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-linear-single",
        task=ExtractionTask(image_bytes=png, x_range=(0, 10), y_range=(0, 10)),
        ground_truth=[Curve(x_values=(0.0, 10.0), y_values=(10.0, 0.0))],
    ))

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 280), pymupdf.Point(280, 20), color=(1, 0, 0), width=2)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(0, 0, 1), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-linear-two-series",
        task=ExtractionTask(image_bytes=png, x_range=(0, 10), y_range=(0, 10)),
        ground_truth=[
            Curve(x_values=(0.0, 10.0), y_values=(0.0, 10.0), series_label="up"),
            Curve(x_values=(0.0, 10.0), y_values=(10.0, 0.0), series_label="down"),
        ],
    ))

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(0, 0, 0), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-log-black-line",
        task=ExtractionTask(image_bytes=png, x_range=(1, 100), y_range=(0, 10), x_scale=ScaleType.LOG),
        ground_truth=[Curve(x_values=(1.0, 100.0), y_values=(10.0, 0.0), x_scale=ScaleType.LOG)],
    ))

    return items


dataset_items = real_items + synthetic_items()
print(f"{len(dataset_items)} total DatasetItem(s)")

## 5. LineFormer ModelRunner adapter

Implements the same `ModelRunnerPort` protocol as `NaiveCvModelRunner` /
`LlmModelRunner` (`extract(task: ExtractionTask) -> list[Curve]`), so it plugs
into the existing `evaluate_model_on_dataset()` usecase unchanged. Pixel→data
calibration reuses `task.x_range`/`task.y_range` exactly like the naive CV
baseline (v0 scope: axis calibration is *given*, not solved by the model —
see design §3.1).

**Runs LineFormer inference via subprocess, not `import`** (design §7.35):
this notebook's kernel stays on whatever Python Colab ships; `torch`/`mmcv`/
`mmdet`/`infer` only exist in the isolated Python 3.10 environment built in
Cell 2 (`PY310`). `extract()` writes the task's image to a temp file, shells
out to a small standalone worker script running under `PY310` (defined below,
written to disk once), and reads back a JSON result file. Each call reloads
the model in a fresh subprocess — slower than loading once and reusing, but
one figure's inference failure can never corrupt or block any other figure's
evaluation, and the notebook kernel never needs `torch`/`cv2`/`mmcv` installed
at all (only `Pillow`, to read the image's pixel dimensions).

In [ ]:
%pip install -q pillow

import io
import json
import pathlib
import subprocess
import tempfile

from PIL import Image

from real_chart_bench.domain.curve import Curve
from real_chart_bench.usecase.model_runner import ExtractionTask

WORKER_SCRIPT_PATH = "/content/lineformer_infer_worker.py"

# Standalone script, run under PY310 (Cell 2) via subprocess -- NOT imported
# into this notebook's own kernel process, which never has mmcv/mmdet/torch
# installed (design §7.35). Takes everything via argv so it needs no shared
# state with the kernel beyond the files it's pointed at.
_WORKER_SCRIPT_SOURCE = '''\
"""LineFormer inference worker -- runs under the isolated Python 3.10 env
(see notebooks/lineformer_colab.ipynb Cell 2), one process per figure.
Reads one image, writes one JSON result file, exits.
"""
import argparse
import json
import sys

sys.path.insert(0, "/content/LineFormer")

import cv2  # noqa: E402
import torch  # noqa: E402
import infer as lineformer_infer  # noqa: E402


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--image", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    device = args.device
    if device == "auto":
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    img = cv2.imread(args.image)
    if img is None:
        raise RuntimeError(f"cv2.imread returned None for {args.image!r}")

    # Verified against LineFormer's README inference example (fetched
    # 2026-08-25): load_model() takes 3 positional args (config, checkpoint,
    # device), not 2 -- an earlier version of this notebook got this wrong.
    lineformer_infer.load_model(args.config, args.checkpoint, device)

    # README's literal example uses to_clean=False; this deliberately uses
    # True instead (LineFormer's own post-processing/cleanup of the
    # extracted line), which should only improve quality here -- not a hard
    # API requirement, just a chosen default.
    series_list = lineformer_infer.get_dataseries(img, to_clean=True)

    # JSON-safe: series_list is pixel-space points per detected line: cast
    # away numpy scalar types.
    payload = [[[float(x), float(y)] for x, y in points] for points in series_list]
    with open(args.output, "w") as f:
        json.dump(payload, f)


if __name__ == "__main__":
    main()
'''

pathlib.Path(WORKER_SCRIPT_PATH).write_text(_WORKER_SCRIPT_SOURCE)


class LineFormerModelRunner:
    """Wraps LineFormer's pretrained instance-segmentation model behind the
    project's ModelRunnerPort protocol. Runs the actual model in a
    subprocess under an isolated Python 3.10 environment (design §7.35 --
    this notebook's own kernel stays on Colab's default Python and never
    imports torch/mmcv/mmdet directly). LineFormer returns pixel-space line
    traces per detected series; this maps each pixel trace to data space
    using the task's given axis calibration (linear or log, independently
    per axis — design §7.25)."""

    def __init__(
        self,
        config_path: str,
        checkpoint_path: str,
        python_path: str,
        worker_script_path: str = WORKER_SCRIPT_PATH,
        device: str = "auto",
    ):
        self._config_path = config_path
        self._checkpoint_path = checkpoint_path
        self._python_path = python_path
        self._worker_script_path = worker_script_path
        self._device = device

    @staticmethod
    def _scale_frac(frac: float, lo: float, hi: float, is_log: bool) -> float:
        # Mirrors domain/pixel_calibration.py's PixelCalibration._scale_frac
        # (design §7.25) -- kept as a small local copy here since this cell
        # runs standalone against a bare pip-installed real_chart_bench, not
        # a dev checkout with test-only helpers exposed.
        if is_log:
            return lo * (hi / lo) ** frac
        return lo + frac * (hi - lo)

    def extract(self, task: ExtractionTask) -> list[Curve]:
        width, height = Image.open(io.BytesIO(task.image_bytes)).size

        with tempfile.TemporaryDirectory() as tmpdir:
            image_path = pathlib.Path(tmpdir) / "input.png"
            output_path = pathlib.Path(tmpdir) / "output.json"
            image_path.write_bytes(task.image_bytes)

            result = subprocess.run(
                [
                    self._python_path,
                    self._worker_script_path,
                    "--config", self._config_path,
                    "--checkpoint", self._checkpoint_path,
                    "--device", self._device,
                    "--image", str(image_path),
                    "--output", str(output_path),
                ],
                cwd="/content/LineFormer",
                capture_output=True,
                text=True,
                timeout=600,
            )
            if result.returncode != 0:
                tail = (result.stderr.strip().splitlines() or ["(no stderr)"])[-1]
                raise RuntimeError(f"LineFormer worker subprocess failed: {tail}")

            series_list = json.loads(output_path.read_text())

        x0, x1 = task.x_range
        y0, y1 = task.y_range
        x_is_log = task.x_scale.name == "LOG"
        y_is_log = task.y_scale.name == "LOG"
        curves = []
        for i, points in enumerate(series_list):
            xs, ys = [], []
            for px, py in points:
                frac_x = px / width
                frac_y = 1.0 - (py / height)  # image y grows downward
                xs.append(self._scale_frac(frac_x, x0, x1, x_is_log))
                ys.append(self._scale_frac(frac_y, y0, y1, y_is_log))
            order = sorted(range(len(xs)), key=lambda j: xs[j])
            xs = [xs[j] for j in order]
            ys = [ys[j] for j in order]
            curves.append(Curve(x_values=tuple(xs), y_values=tuple(ys), series_label=f"series_{i}", x_scale=task.x_scale))
        return curves

## 6. Run evaluation and write results/lineformer-pretrained.json

In [ ]:
from datetime import UTC, datetime

from real_chart_bench.domain.matching import HungarianCurveMatcher
from real_chart_bench.domain.metrics import NormalizedYDistanceMetric
from real_chart_bench.usecase.evaluate_dataset import evaluate_model_on_dataset

model = LineFormerModelRunner(CONFIG_PATH, CHECKPOINT_PATH, PY310)  # device auto-detected inside the worker subprocess
matcher = HungarianCurveMatcher(metric=NormalizedYDistanceMetric())
results = evaluate_model_on_dataset(model, dataset_items, matcher=matcher)

per_figure = [
    {
        "figure_id": r.figure_id,
        "summary_score": r.evaluation.summary_score,
        "match_rate": r.evaluation.match_rate,
        "mean_curve_distance": r.evaluation.mean_curve_distance,
        "mean_coverage_ratio": r.evaluation.mean_coverage_ratio,
        "error": r.error,
    }
    for r in results
]
mean_score = sum(p["summary_score"] for p in per_figure) / len(per_figure)

# dataset_version is derived from the actual evaluated real-pairing count,
# never hardcoded (design §7.28/§7.33: a hardcoded version string went
# stale across the registry's 1->10->20->30-pair expansions and could be
# confused with older runs -- scripts/eval/run_baselines.py uses the same
# derivation).
payload = {
    "model_id": "lineformer-pretrained",
    "model_name": "LineFormer (pretrained, ICDAR2023)",
    "dataset_version": f"v0-eval-pilot-n{len(real_items)}",
    "run_at": datetime.now(UTC).isoformat(),
    "n_figures": len(per_figure),
    "mean_summary_score": mean_score,
    "n_verified_pairs_evaluated": len(real_items),
    "n_verified_pairs_skipped": len(skipped),
    "skipped_pairs": skipped,
    "per_figure": per_figure,
}

out_path = pathlib.Path("/content/lineformer-pretrained.json")
out_path.write_text(json.dumps(payload, indent=2))

print("=" * 70)
print(
    f"SUMMARY: evaluated {len(real_items)} verified real pairing(s) "
    f"+ {len(dataset_items) - len(real_items)} synthetic fixture(s) "
    f"= {len(per_figure)} figure(s) scored."
)
print(f"         skipped {len(skipped)} verified pairing(s) (excluded from scoring):")
for s in skipped:
    print(f"           paper {s['paper_id']} figure {s['figure_id']}: {s['reason']}")
print(f"mean_summary_score = {mean_score:.4f}, computed over the {len(per_figure)} figure(s) above")
print("=" * 70)
print(json.dumps(payload, indent=2))

## 7. Download the result and add it to the repo

This notebook does **not** push to git automatically (same structural-non-
execution pattern as the HF Hub upload guard and the LLM adapter — pushing
results is a deliberate, reviewed action, not a side effect of running a
notebook). Download the file, then on your own machine:

```bash
mv ~/Downloads/lineformer-pretrained.json results/lineformer-pretrained.json
rm results/lineformer-pending.json
python scripts/leaderboard/generate.py
git add results/ site/
git commit -m "results: LineFormer pretrained baseline (Colab run)"
git push origin main
```

In [ ]:
from google.colab import files
files.download(str(out_path))